### Step 1: Determine the translation speed $c$
Simulate the Gray–Scott model to obtain $X_0(x-\Phi, t)$.

In [ ]:
import sys 
if 'ipykernel' in sys.modules:
    # Enable inline plotting only when running in Jupyter.
    get_ipython().run_line_magic('matplotlib', 'inline')
    get_ipython().run_line_magic('config', "InlineBackend.figure_formats = {'png', 'retina'}")


import numpy as np
from numpy import random
import numpy.linalg as LA
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import importlib
from typing import Literal
from tqdm import tqdm
import pickle, gc


import ModuleGrayScottModel_Yadome as GSM_Yadome
import ModuleGrayScottModel_JIT_simulation as GSM_Yadome_JIT
from functions.intersect_time import intersect_time
importlib.reload(GSM_Yadome)

import os
sys.path.append("functions")

from time_deco import log_execution_time
from SpectralDecomposition import calculate_Hk, calculate_A
from measure_period import return_to_section, measure_period
from ndarray_manipulate import myfunc_roll
from functions.myfunc_interp_numpy import myfunc_interp_numpy
from functions.intersect_time import intersect_time


dict_property = {
    "dt": 0.01,
    "STEP": int(1),
    "ITERATION": int(50),
    "LENGTH": float(5000),
    "gridnum_in_cycle": int(2**20+1) #* 
}


In [ ]:
ITERATION = dict_property["ITERATION"]
LENGTH = dict_property["LENGTH"]
STEP = dict_property["STEP"]

filename_postfix = "debug"

# %% 
def calculate_trajectory(GSM, x0, tspan, key:Literal["x", "chi"], step=int(1)):
    GSM_jit = GSM_Yadome_JIT.GrayScottModel_JIT(GSM)
    if key =="x":
        U, V, tsave = GSM_jit.pde(x0, tspan, step=step)
    elif key == "chi":
        U, V, tsave = GSM_jit.pde_with_chi(x0, tspan, step=step)
    else:
        raise ValueError("key must be 'x' or 'chi'")
    return [U, V], tsave

In [ ]:
import os
from pathlib import Path

delete_on = False  # Set to True to delete the listed files.
targets = [
    "freqdict.npz",
    "trajectory_GSM_chi-space.npz",
    "trajectory_GSM_x-space.npz",
    "X0_1025.pickle",
]

base_dir = Path.cwd()

for name in targets:
    path = base_dir / name
    if not path.exists():
        print(f"skip: {path} does not exist")
        continue

    if delete_on:
        try:
            path.unlink()
            print(f"deleted: {path}")
        except Exception as exc:
            print(f"failed: {path} ({exc})")
    else:
        print(f"kept (delete_off): {path}")


In [ ]:
GSM = GSM_Yadome.GrayScottModel()
x0 = GSM.initialize_x0(center=0.0)


In [ ]:
# Load or compute the transient trajectory.
filename = "./trajectory_GSM_x-space.npz".format(filename_postfix)
if not(os.path.isfile(filename)):
    ts = 0.0
    dt = dict_property["dt"]
    for i in tqdm(range(ITERATION), desc="iteration", ncols=60):
        taxis = np.arange(ts, ts+LENGTH+dt, dt)
        [U, V], tsave = calculate_trajectory(GSM, x0, taxis, key="x", step=STEP)
        x0 = [U[-1], V[-1]]
        ts = tsave[-1]

        if i < (ITERATION - 1):
            del U, V
            gc.collect()
            
    np.savez(filename, U=U, V=V, tsave=tsave)
    print("save data ... \n {:s}".format(filename))

else:
    print("load data ... \n {:s}".format(filename))
    data1 = np.load(filename)
    U, V, tsave = data1["U"], data1["V"], data1["tsave"]
    del data1


data1 = {"U": U, "V":V,
         "t":tsave, "rx":GSM.rx}

del U, V, tsave
gc.collect()

Colormap

In [ ]:
U = data1["U"]; V = data1["V"]; 
rx = data1["rx"]; t = data1["t"]

arg = (t[0]<=t) & (t<=(t[0]+5000))
s = 2**7
U = U[arg][::s]; V = V[arg][::s]; 
rx = rx; t = t[arg][::s]


#* plot colormap
fig, axes = plt.subplots(ncols=2, figsize=(6, 3))
##
ax1 = axes[0]
cax1 = ax1.imshow(U, aspect='auto', cmap='viridis', 
                  extent=[rx.min(), rx.max(), t[0], t[-1]], origin='lower')
ax1.set_title('$u (x, t; \Phi=ct+\phi_0)$', fontsize=11)
fig.colorbar(cax1, ax=ax1)

##
ax2 = axes[1]
cax2 = ax2.imshow(V, aspect='auto', cmap='viridis', 
                  extent=[rx.min(), rx.max(), t[0], t[-1]], origin='lower')
ax2.set_title('$v (x, t; \Phi=ct+\phi_0)$', fontsize=11)
fig.colorbar(cax2, ax=ax2)

fig.tight_layout()


del U, V, t, rx

Determine the angular frequency and period $T_t$ of the temporal phase $\Theta$.

$\overline{u} = \int_{L/2}^{L/2} u(x, \Theta; \Phi) dx$

In [ ]:

#* plot Hu0, Hv0 to determine the value of Poincaré section set on Hu0 or Hv0
U = data1["U"]; V = data1["V"]; 
rx = data1["rx"]; t = data1["t"]
arg = (t[0]<=t)&(t<=t[0]+1000)

U = U[arg]; V = V[arg]; t = t[arg]
Hu0 = calculate_Hk(U, rx, j=0)
Hv0 = calculate_Hk(V, rx, j=0)

# Use this plot to select the Poincare-section level.
plt.figure(figsize=(8,2))
plt.subplot(121)
plt.plot(t, Hu0.real)
plt.ylabel("$\overline{u}$")
plt.subplot(122)
plt.plot(t, Hv0.real)
plt.ylabel("$\overline{v}$")
plt.tight_layout()


### Settings for determining the temporal-phase period

In [ ]:

# Poincare section used to measure one period of Theta.
dict_Poincate = {"uv_key":"U",
                  "section": 7.0,
                   "index": int(0)}

T_GRIDNUM = int(2**20+1)
SAVE_STEP = int(2**6)

In [ ]:
@log_execution_time
def cal_onecycle(x0, DT=float(220/2**20), LENGTH=float(220), 
                   c=float(0.0)):
    """
    x0 = [u, v]
    LENGTH is the integration period.
    A nonzero c selects chi coordinates; c = 0 selects x coordinates.
    """
    RX_GRIDNUM = int(x0[0].size)

    # Set the spatial-grid size.
    paramdict = {
        "gridnum": RX_GRIDNUM
    }
    # Integrate one cycle.
    GSM = GSM_Yadome.GrayScottModel()
    tspan = np.arange(0, LENGTH+DT, DT)
    [u,v], tsave = calculate_trajectory(GSM, x0, tspan, key="x", step=int(2**4))
    rx = GSM.rx
    del GSM

    # Report the time-grid mismatch.
    print("DT*iteration={:0.6f}".format(DT*round(LENGTH/DT)))
    print("LENGTH={:0.6f}".format(LENGTH))
    print("Difference: {:0.12f}".format(DT*round(LENGTH/DT)-LENGTH))

    return [u, v], tsave, rx



## Determine the temporal-phase period

In [ ]:
# Determine omega.

U, V, rx, t = data1["U"], data1["V"], data1["rx"], data1["t"]

u_bar = np.trapz(U, rx, axis=1)
periods = np.diff(intersect_time(t=t, x=u_bar, section=dict_Poincate["section"]))
T = np.mean(periods)
omega = (2.0*np.pi/T)


print("*-------------------------------------------------------------*")
print("Temporal-phase period T = {:0.6e}".format(T))
print("Temporal-phase angular frequency = {:0.6e}".format(omega))

print("Details:")
print("Mean period T: {:0.6e}; standard deviation: {:0.6e}".format(periods.mean(), periods.std()))
print("Mean angular frequency omega: {:0.6e}; standard deviation: {:0.6e}".format((2.0*np.pi/periods).mean(), (2.0*np.pi/periods).std()))

# End of omega calculation.

## 

Check convergence of the period $T$.

In [ ]:
from matplotlib.ticker import ScalarFormatter
plt.figure(figsize=(4,2))
plt.plot(range(len(periods)), periods)

formatter = ScalarFormatter(useOffset=False)
plt.gca().yaxis.set_major_formatter(formatter)
if np.std(periods) <= 1e-4:
    plt.ylim(np.mean(periods)-1e-3, np.mean(periods)+1e-3)
plt.xlabel("cycle")
plt.ylabel("$T_i$")
plt.tight_layout()

print("Mean: {:0.4e}".format(np.mean(periods)))
print("Standard deviation: {:0.4e}".format(np.std(periods, ddof=1)))
print("[min, max]=[{:0.12e}, {:0.12e}]".format(np.min(periods), np.max(periods)))

## Determine the translation speed $c$

In [ ]:
dict_Freq = {
                "omega": omega, "T_theta": T,
                "c": None, "T_phi": None, 
                "ref_T": periods,
                "ref_c": None, "ref_arg_A": None
            }

DT = T / (T_GRIDNUM-1)
ITERATION_NUM = int(51)

Determine $c$ by solving the following equation:
$$
\frac{L}{2\pi} \left[ \mathrm{arg} A(ncT_t, 2n\pi) - \mathrm{arg} A(c(n-1)T_t, 2(n-1)\pi) \right]
= c T
\tag{1}
$$

Starting from the reference state at $\Theta=0$, compute $\bm{X}_0(x-\Phi, t)$ and evaluate $A(\Phi(nT_t), \Theta(nT_t)[=2\pi n])$ after evolving for $nT$.

In [ ]:
import contextlib, io

# Load or compute the transient trajectory.
filename = "./freqdict.npz"

if not(os.path.isfile(filename)):
    # Advance to the Poincare section.
    u_bar = np.trapz(U, rx, axis=1)
    times = intersect_time(t=t, x=u_bar, section=dict_Poincate["section"])

    print("Time range: {:0.6f} to {:0.6f}".format(t[0], t[-1]))
    print("Interpolation time: {:0.6f}".format(times[-1]))

    u0 = myfunc_interp_numpy(times[-1], t, U)
    v0 = myfunc_interp_numpy(times[-1], t, V)
    old_x0 = (u0, v0)

    list_A = [] 
    for i in tqdm(range(ITERATION_NUM)):
        with contextlib.redirect_stdout(io.StringIO()):
            # Repeatedly integrate forward by T.
            new_x0, tsave, rx = cal_onecycle(x0=old_x0, DT=DT, LENGTH=dict_Freq["T_theta"],
                                            c=0.0)
            old_x0 = [new_x0[0][-1], new_x0[1][-1]]  # Final state.
            # Evaluate A(x0[nT]).
            u0 = new_x0[0][-1]  # Final state.
            list_A.append(calculate_A(u0.reshape([1,-1]), rx))
    list_A = np.array(list_A).ravel()

    # Compute the unwrapped argument of A(nT).
    arg_A = np.unwrap(np.mod(np.angle(list_A), 2.0*np.pi))

    # Compute c.
    c_list = []
    for i in range(1, len(arg_A)):
        T = dict_Freq["T_theta"]
        L = np.ptp(rx)
        c = (L/(2.0*np.pi)) * (arg_A[i]-arg_A[i-1]) / T
        c_list.append(c)
    c_list = np.array(c_list)

    # Store the frequency estimates.
    dict_Freq["c"] = c_list.mean()
    dict_Freq["T_phi"] = np.ptp(rx) / dict_Freq["c"]
    dict_Freq["ref_c"] = c_list
    dict_Freq["ref_arg_A"] = arg_A

    np.savez(filename, **dict_Freq)
    print("save data ... \n {:s}".format(filename))

else:
    print("load data ... \n {:s}".format(filename))
    datas = np.load(filename)
    c_list = datas["ref_c"]
    dict_Freq = datas
    

print("*-------------------------------------------------------------*")
print("Number of c estimates: {:d}".format(c_list.size))
print("Mean c: {:0.6e}".format(c_list.mean()))
print("Standard deviation of c: {:0.6e}".format(c_list.std()))
print("Maximum c: {:0.6e}".format(c_list.max()))
print("Minimum c: {:0.6e}".format(c_list.min()))



**Check convergence of $c$.**  



In [ ]:
plt.figure(figsize=(7,3))
plt.subplot(121)
plt.plot(range(len(dict_Freq["ref_c"])), dict_Freq["ref_c"])
plt.title("$\{c_n\}_{n=1}^{1000}$")
plt.xlabel("$n$ (cycle)")

plt.subplot(122)
plt.plot(range(len(dict_Freq["ref_c"])), 250/dict_Freq["ref_c"])
plt.title("period: $\{2L / c_n\}_{n=1}^{1000}$")
plt.xlabel("$n$ cycle")

plt.tight_layout()

print("c mean:{:0.4e}".format(np.mean(c_list)))
print("c std:{:0.4e}".format(np.std(c_list, ddof=1)))

print("period mean:{:0.4e}".format( np.mean(250/dict_Freq["ref_c"])  ))
print("period std:{:0.4e}".format( np.std(250/dict_Freq["ref_c"], ddof=1) ))



Solving the translation-speed equation by least squares gives $c = \langle c_n \rangle_n$.  
If convergence is slow, use the estimates from the latter part of the sequence.

### Step 2: Simulate the Gray–Scott model
With the coordinate transformation $\chi=x-ct$, the equation is
$$
\partial_x \tilde{\bm{X}}(\chi,\tau) = \bm{F}(\bm{X}) + D \partial_x^2 \tilde{\bm{X}} + c\partial_x \tilde{\bm{X}}
$$






In [ ]:
GSM.c = dict_Freq["c"]
chi0 = GSM.initialize_x0(center=0.0)  # Initial center.

In [ ]:

# Load or compute the transient trajectory.
filename = "./trajectory_GSM_chi-space.npz"


if not(os.path.isfile(filename)):
    ts = 0.0
    dt = dict_property["dt"]
    for i in tqdm(range(ITERATION), desc="iteration", ncols=60):
        taxis = np.arange(ts, ts+LENGTH+dt, dt)
        [U, V], tsave = calculate_trajectory(GSM, chi0, taxis, key="chi", step=STEP)
        chi0 = [U[-1], V[-1]]
        ts = tsave[-1]

        if i < (ITERATION - 1):
            del U, V
            gc.collect()
        
    np.savez(filename, U=U, V=V, tsave=tsave)
    print("save data ... \n {:s}".format(filename))

else:
    print("load data ... \n {:s}".format(filename))
    data2 = np.load(filename)
    U, V, tsave = data2["U"], data2["V"], data2["tsave"]
    del data2

data2 = {"U": U, "V":V,
         "t":tsave, "rx":GSM.rx}

del U, V, tsave
gc.collect()


In [ ]:
U = data2["U"]; V = data2["V"]; 
rx = data2["rx"]; t = data2["t"]

arg = (t[0]<=t) & (t<=(t[0]+5000))

s = 2**7
U = U[arg][::s]; V = V[arg][::s]; 
rx = rx; t = t[arg][::s]


#* plot colormap
fig, axes = plt.subplots(ncols=2, figsize=(6, 3))
##
ax1 = axes[0]
cax1 = ax1.imshow(U, aspect='auto', cmap='viridis', 
                  extent=[rx.min(), rx.max(), t[0], t[-1]], origin='lower')
ax1.set_title(r"$\tilde{u} (\chi, t; \Phi' = \phi_0)$", fontsize=11)
fig.colorbar(cax1, ax=ax1)

##
ax2 = axes[1]
cax2 = ax2.imshow(V, aspect='auto', cmap='viridis', 
                  extent=[rx.min(), rx.max(), t[0], t[-1]], origin='lower')
ax2.set_title(r"$\tilde{v} (\chi, t; \Phi' = \phi_0)$", fontsize=11)
fig.colorbar(cax2, ax=ax2)

fig.tight_layout()


del U, V, t, rx

## Obtain $\bm{X}_0(\chi, t)$

Extract $\tilde{\bm{X}}_0(\chi, t)$ over one $2\pi$ increase of $\Theta$.

In [ ]:
# get the state on the Poincaré section
x0, _ = return_to_section([data2["U"], data2["V"]], data2["rx"], data2["t"], 
                section=dict_Poincate["section"], 
                key=dict_Poincate["uv_key"], 
                index=dict_Poincate["index"]
                )

# The period of Theta is known.
T = dict_Freq["T_theta"]

# Integrate one cycle.
tgridnum = dict_property["gridnum_in_cycle"]; 
tl = np.linspace(0, T, num=int(tgridnum))
[Ul, Vl], _ = calculate_trajectory(GSM, x0, tl, key="chi")

data_cycle = {
            "U": Ul[::int(2**4)], "V": Vl[::int(2**4)],
              "rx": GSM.rx, 
            "t": tl[::int(2**4)], "theta": 2.*np.pi*(tl[::int(2**4)]/T),
            "phi":np.zeros_like(tl), "c":dict_Freq["c"],
            }

with open('X0_{:d}.pickle'.format(GSM.rx.size), 
          mode='wb') as f:
    pickle.dump(data_cycle , f)

In [ ]:
#* plot colormap of one-cycle 
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


U = data_cycle["U"]; V = data_cycle["V"]; 
rx = data_cycle["rx"]; theta = data_cycle["theta"]; t = data_cycle["t"]
L = 0.5*np.ptp(rx) 
s = max(1, (U.shape[0] - 1) // int(2**10 - 1))


U = U[::s]; V = V[::s]; 
theta = theta[::s]
t = t[::s]
rx_axis = rx / (0.5*np.ptp(rx)) #* -1~1
t_axis = theta / np.ptp(theta) * 2.0 #* 0~2



In [ ]:

fig = plt.figure(figsize=(7, 3))
fig.subplots_adjust(wspace=0.8)
# Colormap of u0.
ax = fig.add_subplot(121)
c1 = ax.pcolormesh(rx_axis, t_axis, U,
                    cmap='viridis', 
                    vmin=0.0, vmax=0.32, rasterized=True)
# Create a vertical colorbar on the right of the main plot
cax = inset_axes(
    ax,
    width="5%",
    height="100%",
    loc='center right',
    bbox_to_anchor=(0.1, 0, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=0
)
fig.colorbar(c1, cax=cax, orientation='vertical')

ax.set_xlabel(r'$x/L$')
ax.set_ylabel(r'$\Theta/\pi$')    
ax.set_xticks([-1, 0, 1])
ax.set_yticks([0, 1, 2])
ax.set_aspect(np.ptp(rx_axis) / np.ptp(t_axis)) 
ax.set_title(r'$u_0(x-\Phi, \Theta) ~[\Phi=0]$')


# Colormap of v0.
bx = fig.add_subplot(122)
c2 = bx.pcolormesh(rx_axis, t_axis, V,
                    cmap='viridis', 
                    vmin=0.2, vmax=1., rasterized=True)
# Create a vertical colorbar on the right of the main plot
cbx = inset_axes(
    bx,
    width="5%",
    height="100%",
    loc='center right',
    bbox_to_anchor=(0.1, 0, 1, 1),
    bbox_transform=bx.transAxes,
    borderpad=0
)
fig.colorbar(c2, cax=cbx, orientation='vertical')

bx.set_xlabel(r'$x/L$')
bx.set_ylabel(r'$\Theta/\pi$')    
bx.set_xticks([-1, 0, 1])
bx.set_yticks([0, 1, 2])
bx.set_aspect(np.ptp(rx_axis) / np.ptp(t_axis)) 
bx.set_title(r'$v_0(x-\Phi, \Theta) ~[\Phi=0]$')

plt.show()





### Determine the function $B$
$$
B[\Theta(t)]  = \frac{L}{\pi} \mathrm{arg} \tilde{A}[\Phi'(t), \Theta(t)] \\
$$






In [ ]:
from functions.SpectralDecomposition import calculate_A
from scipy.interpolate import interp1d

def get_Bfunc_from_X0(X0):
    """
    Construct the periodic function B from X0 using
    B[\Theta(t)] + \Phi' = \frac{L}{2 \pi} \mathrm{arg} \tilde{A}[\Phi'(t), \Theta(t)].
    The constant term on the right is Phi'; subtract it from the left
    and shift X0 and related functions by the same amount.
    """
    # Compute arg(A) from the U time series.
    argA = np.mod(np.angle(calculate_A(X0["U"], X0["rx"])), 2.0*np.pi)
    # B + Phi' = (L/2pi)*argA; set Phi' = 0 temporarily.
    B = (np.ptp(X0["rx"])/(2.0*np.pi)) * argA
    # Interpolate B periodically.
    _B_interp = interp1d(X0["theta"], B, kind="linear")
    B_interp = lambda theta: _B_interp(np.mod(theta, 2.0*np.pi))
    return B_interp


In [ ]:
with open('X0_1025.pickle', mode='rb') as f:
    a = pickle.load(f)
X0 = {
        "U": a["U"], "V": a["V"],
        "rx": a["rx"], "t": a["t"],
        "theta": 2.0*np.pi * (a["t"]-a["t"][0]) / np.ptp(a["t"]),
    }

# Compute the spatial means of U and V in X0.
X0["ubar"] = np.trapz(X0["U"], X0["rx"], axis=1)
X0["vbar"] = np.trapz(X0["V"], X0["rx"], axis=1)
ubar_func = interp1d(X0["theta"], X0["ubar"], kind="linear")
vbar_func = interp1d(X0["theta"], X0["vbar"], kind="linear")
# Compute B(Theta).
Bfunc = get_Bfunc_from_X0(X0)
B_mean = np.trapz(Bfunc(X0["theta"]), X0["theta"]) / (2.0*np.pi)


# Save B.
B_dict = {
        "By": Bfunc(X0["theta"])-B_mean,
        "Bx": X0["theta"],
        "Bmean": B_mean
    }
np.savez_compressed('B_func.npz', **B_dict)

Plot the resulting function $B$.  
The relation between $B$ and $A$ shows that, in $\chi$ coordinates, oscillations of $B(\Theta)$ appear directly in $\mathrm{arg}A$.

In [ ]:
plt.figure(figsize=(3.4,2))
plt.plot(B_dict["Bx"], B_dict["By"], color="k")
# plt.plot(np.linspace(0, 4.0*np.pi, num=1024), B_interp(np.linspace(0, 4.0*np.pi, num=1024)), color="k")
# plt.plot(theta, B)
# plt.xlim(0, 2.0*np.pi)
plt.ylabel("$B(\Theta)$")
plt.xticks(np.linspace(0, 2.0*np.pi, 3),
           labels=["$0$", "$\pi$", "$2.0\pi$"])
plt.ylim(-1,1)
plt.grid(linestyle="-.")


## Shift $X_0$ spatially by the constant term of $B$

In [ ]:
def shift_matrix(x, A, shift=B_mean):
    """ 
    Translate (x, y, A) by ``shift`` along x.
    x: rx/2L (-1~1)
    A: 2d-array
    shift: Translation distance.
    """
    L = 250  # Spatial-domain length.
    shift = shift / (0.5*L)  # Dimensionless translation (= Phi).
    print(shift)
    if shift == 0:
        return A
    else:
        _A_func = interp1d(x, A, kind='linear', axis=1)

        def my_mod(x):
            """Wrap x into [-1, 1]."""
            _x = np.mod(x,2)
            _x[_x > 1] -= 2
            return _x
        
        # This convention translates (x, A) by -shift when plotted.
        return _A_func(my_mod(x+shift))
    

In [ ]:
#* plot colormap of one-cycle 

U = data_cycle["U"]; V = data_cycle["V"]; 
rx = data_cycle["rx"]; theta = data_cycle["theta"]; t = data_cycle["t"]
L = 0.5*np.ptp(rx) 
s = max(1, (U.shape[0] - 1) // int(2**10 - 1))


U = U[::s]; V = V[::s]; 
theta = theta[::s]
t = t[::s]
rx_axis = rx / (0.5*np.ptp(rx)) #* -1~1
t_axis = theta / np.ptp(theta) * 2.0 #* 0~2


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig = plt.figure(figsize=(7, 3))
fig.subplots_adjust(wspace=0.8)
# Colormap of u0.
ax = fig.add_subplot(121)
c1 = ax.pcolormesh(rx_axis, t_axis, 
                   shift_matrix(rx_axis, U),
                    cmap='viridis', 
                    vmin=0.0, vmax=0.32, rasterized=True)
# Create a vertical colorbar on the right of the main plot
cax = inset_axes(
    ax,
    width="5%",
    height="100%",
    loc='center right',
    bbox_to_anchor=(0.1, 0, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=0
)
fig.colorbar(c1, cax=cax, orientation='vertical')

ax.set_xlabel(r'$x/L$')
ax.set_ylabel(r'$\Theta/\pi$')    
ax.set_xticks([-1, 0, 1])
ax.set_yticks([0, 1, 2])
ax.set_aspect(np.ptp(rx_axis) / np.ptp(t_axis)) 
ax.set_title(r'$u_0(x-\Phi, \Theta) ~[\Phi=0]$')


# Colormap of v0.
bx = fig.add_subplot(122)
c2 = bx.pcolormesh(rx_axis, t_axis,
                   shift_matrix(rx_axis, V),
                    cmap='viridis', 
                    vmin=0.2, vmax=1., rasterized=True)
# Create a vertical colorbar on the right of the main plot
cbx = inset_axes(
    bx,
    width="5%",
    height="100%",
    loc='center right',
    bbox_to_anchor=(0.1, 0, 1, 1),
    bbox_transform=bx.transAxes,
    borderpad=0
)
fig.colorbar(c2, cax=cbx, orientation='vertical')

bx.set_xlabel(r'$x/L$')
bx.set_ylabel(r'$\Theta/\pi$')    
bx.set_xticks([-1, 0, 1])
bx.set_yticks([0, 1, 2])
bx.set_aspect(np.ptp(rx_axis) / np.ptp(t_axis)) 
bx.set_title(r'$v_0(x-\Phi, \Theta) ~[\Phi=0]$')

plt.show()


### Save the final $X_0$


In [ ]:
U = data_cycle["U"]; V = data_cycle["V"]; 
rx = data_cycle["rx"]; theta = data_cycle["theta"]; t = data_cycle["t"]
L = 0.5*np.ptp(rx) 
T = np.ptp(t)
s = max(1, int((U.shape[0] - 1) / int(2**10)) )

U = U[::s]; V = V[::s]; 
U = shift_matrix(rx_axis, U)
V = shift_matrix(rx_axis, V)
theta = theta[::s]
t = t[::s]
rx_axis = rx  #* -1~1
t_axis = t #* 0~2


X0withPhi0 =  {"U":U, "V":V,
         "t":t_axis, "rx":rx_axis,
         "T":np.ptp(t)
         }


np.savez_compressed('X0data_withPhi0.npz', **X0withPhi0)

In [ ]:
X0withPhi0["U"].shape

In [ ]:
print(X0withPhi0 )

In [ ]:
print(X0withPhi0["t"])

In [ ]:
print(X0withPhi0["rx"])